In [1]:
import random

import pandas as pd
from pandas import DataFrame
import numpy as np

import math

from preferences import prefs
from user import user_prefs

# startPoint = {"lat": 30.317504,"lon": 59.927085}
# endPoint = {"lat": 30.327108, "lon": 59.935408}

POPULATION_SIZE = 10
GENERATIONS = 100

MIN_ROUTE_POINTS = 3
MAX_ROUTE_POINTS = 10

KILLOMETER_RADIUS = 0

MUTATION_RATE = 1

PRINT = True

# random.seed(42)
# np.random.seed(42)

# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }


In [2]:
df = pd.read_csv('data/places.csv')

**Алгоритмы генетики**

In [ ]:
from haversine import haversine

# Оптимизация конкретного маршрута
def optimize_route(route_ids, df, startPoint):
    points = (
        df[df["id"].isin(route_ids)]
        [["id", "lat", "lon"]]
        .copy()
    )

    remaining = points.to_dict("records")

    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    optimized_route = []

    while remaining:
        nearest = min(
            remaining,
            key=lambda p: haversine(
                (current["lat"], current["lon"]),
                (p["lat"], p["lon"])
            )
        )

        optimized_route.append(nearest["id"])

        current = nearest

        remaining.remove(nearest)

    return optimized_route
    

# Генерация случайной особи
def create_population(df: DataFrame, startPoint: dict[str, float], endPoint: dict[str, float], num: int):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    buffer_km = KILLOMETER_RADIUS

    # Переводим километры в градусы
    lat_buffer = buffer_km / 111

    mean_lat = (startPoint["lat"] + endPoint["lat"]) / 2
    lon_buffer = buffer_km / (111 * np.cos(np.radians(mean_lat)))

    min_lat = min(startPoint["lat"], endPoint["lat"]) - lat_buffer
    max_lat = max(startPoint["lat"], endPoint["lat"]) + lat_buffer

    min_lon = min(startPoint["lon"], endPoint["lon"]) - lon_buffer
    max_lon = max(startPoint["lon"], endPoint["lon"]) + lon_buffer

    df_filtered = df[
        (df["lat"] >= min_lat) &
        (df["lat"] <= max_lat) &
        (df["lon"] >= min_lon) &
        (df["lon"] <= max_lon)
    ]

    population = []

    df_indexes = df_filtered["id"].tolist()

    for _ in range(num):
        route_id = random.sample(
            range(len(df_indexes)),
            route_size
        )
        route = []

        for i in route_id:
            route.append(df_indexes[i])
        
        new_route = optimize_route(route, df, startPoint)
        
        population.append({"route": new_route, "fitness": None})

    return population

# Длина маршрута
def route_distance(route: list, startPoint: dict[str, float], endPoint: dict[str, float]):
    
    route_df = pd.DataFrame()

    for j in route:
        route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

    remaining = route_df.to_dict(orient='index')
    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    distance = haversine((current["lat"], current["lon"]),
                         (remaining[0]["lat"], remaining[0]["lon"]))

    current = remaining[0]

    for point_id in remaining.keys():
        if point_id == 0:
            continue
        distance += haversine((current["lat"], current["lon"]),
                              (remaining[point_id]["lat"], remaining[point_id]["lon"]))
        current = remaining[point_id]
    distance = haversine((current["lat"], current["lon"]),
                         (endPoint["lat"], endPoint["lon"]))
    return distance


# Интересность конкретной точки
def point_interest_score(point: dict, unique_user_prefs: dict[str, int]):
    point_tags = next(iter(point.values()))
    score = 0
    for category, tags in prefs.items():
        weight = unique_user_prefs[category]
        for tag in tags:
            if tag in point_tags and point_tags[tag]:
                score += weight
    return score

# Фитнес-функция
def fitness_function(route: list, unique_user_prefs: dict[str, int], startPoint: dict[str, float], endPoint: dict[str, float]):

    total_interest = 0

    for place_id in route:
        point = df.iloc[(df["id"] == place_id)].to_dict(orient='index')
        total_interest += point_interest_score(point, unique_user_prefs)

    distance = route_distance(route, startPoint, endPoint)

    fitness = total_interest * 20 - distance * 0.03

    return fitness

# Проведение оценки популяции (просчёт фитнес-функций каждой особи)
def evaluate_population(population: dict, unique_user_prefs: dict[str, int], startPoint: dict[str, float], endPoint: dict[str, float]):
    for individual in population:
        individual["fitness"] = fitness_function(
            individual["route"],
            unique_user_prefs,
            startPoint,
            endPoint
        )


# Мутация
def mutate(individual, unique_user_prefs: dict[str, int]):
    if random.random() > MUTATION_RATE:
        return individual
    
    route = individual["route"][:]
    
    route_df = pd.DataFrame()

    for j in route:
        route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

    mutation_type = random.choice([
        "remove_minimum_score",
        "remove_most_distant",
        # "add"
    ])


    # print("До мутации:", route)

    if mutation_type == "remove_minimum_score":
        if len(route) > MIN_ROUTE_POINTS:
            print("Удаление минимального по интересности:", route)
            min_score_point = route[0]
            
            min_score = point_interest_score({0:df.iloc[0].to_dict()}, unique_user_prefs)
            for place_id in route:
                new_min_score = point_interest_score({0:df.iloc[(df["id"] == place_id)].to_dict()}, unique_user_prefs)
                if min_score > new_min_score:
                    min_score = new_min_score
                    min_score_point = place_id
            
            route.remove(min_score_point)

    elif mutation_type == "remove_most_distant":
        if len(route) > MIN_ROUTE_POINTS:
            print("Удаление максимального по удалённости:", route)
            sum_lat = 0
            sum_lon = 0
            for index, row in route_df.iterrows():
                sum_lat += row["lat"]
                sum_lon += row["lon"]
            mean_lat = sum_lat / len(route)
            mean_lon = sum_lon / len(route)
            
            max_distance_point = route[0]
            
            max_distance = haversine((mean_lat, mean_lon), (route_df["lat"].iloc[0], route_df["lon"].iloc[0]))
            
            for place_id in route:
                place = route_df.loc[route_df["id"] == place_id].iloc[0]

                new_max_distance = haversine((mean_lat, mean_lon),(place["lat"], place["lon"]))
                
                if max_distance < new_max_distance:
                    max_distance = new_max_distance
                    max_distance_point = place_id
            
            route.remove(max_distance_point)

    
    # print("После мутации:", route)
    
    # elif mutation_type == "add":

    #     if len(route) < MAX_ROUTE_POINTS:

    #         available = list(
    #             set(range(len(df))) - set(route)
    #         )

    #         if available:

    #             count_to_add = random.randint(1, 3)

    #             for _ in range(count_to_add):

    #                 if not available:
    #                     break

    #                 new_point = random.choice(available)
    #                 available.remove(new_point)

    #                 insert_pos = random.randint(
    #                     0,
    #                     len(route)
    #                 )

    #                 route.insert(insert_pos, new_point)

    # individual["route"] = route
    # individual["fitness"] = None

    return {"route": route, "fitness": None}

# Создание нового поколения
def create_next_generation(population: dict, unique_user_prefs: dict[str, int]):
    new_population = []

    population.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    # elite = population[:math.ceil(POPULATION_SIZE/10)]
    elite = population[:5]

    new_population.extend(elite)

    while len(new_population) < POPULATION_SIZE:
        child = mutate(random.choice(elite), unique_user_prefs)
        new_population.append(child)

    return new_population

In [4]:
import folium

def hex_to_rgb(hex_str):
    """Преобразует HEX в кортеж RGB (0-255)"""
    hex_str = hex_str.lstrip('#')
    return tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(rgb):
    """Преобразует RGB в HEX строку"""
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))

def generate_gradient(start_hex, end_hex, steps):
    """Генерирует список цветов для градиента"""
    start_rgb = hex_to_rgb(start_hex)
    end_rgb = hex_to_rgb(end_hex)
    
    gradient_colors = []
    for i in range(steps):
        t = i / max(1, steps - 1)
        r = start_rgb[0] + (end_rgb[0] - start_rgb[0]) * t
        g = start_rgb[1] + (end_rgb[1] - start_rgb[1]) * t
        b = start_rgb[2] + (end_rgb[2] - start_rgb[2]) * t
        gradient_colors.append(rgb_to_hex((r, g, b)))
        
    return gradient_colors

def visual():

    path_colors = generate_gradient("#FF0000","#0000FF", len(population))

    route_df = df[(df["id"].isin(population[0]["route"]))]
    m = folium.Map(
        location=[
            route_df["lat"].mean(),
            route_df["lon"].mean()
        ],
        zoom_start=15
    )

    for i in range(len(population)):
        individual = population[i]
        
        route_df = pd.DataFrame()

        for j in individual["route"]:
            route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

        # Точки маршрута
        folium.Marker(
            [startPoint["lat"], startPoint["lon"]]
        ).add_to(m)
        folium.Marker(
            [endPoint["lat"], endPoint["lon"]]
        ).add_to(m)

        route_line = [[startPoint["lat"], startPoint["lon"]]]
        route_line.extend(route_df[["lat", "lon"]].values.tolist())
        route_line.append([endPoint["lat"], endPoint["lon"]])

        # Линия маршрута
        folium.PolyLine(
            route_line,
            weight=4,
            color=path_colors[i]
        ).add_to(m)

    m.save(r"visualisations\route.html")

**Запуск генетики**

In [6]:
startPoint = {"lat": 59.927085,"lon": 30.317504}
endPoint = {"lat": 59.935408, "lon": 30.327108}

population = create_population(df, startPoint, endPoint, POPULATION_SIZE)

unique_user_prefs = user_prefs

for generation in range(GENERATIONS):
    evaluate_population(population, unique_user_prefs, startPoint, endPoint)

    best = max(
        population,
        key=lambda x: x["fitness"]
    )

    if PRINT:
        print(
            f"Поколение {generation} | "
            f"Лучшая фитнес-функция: {best['fitness']:.10f} | "
            f"Длина маршрута: {len(best['route'])}"
    )
    visual()
    population = create_next_generation(population, unique_user_prefs)

Поколение 0 | Лучшая фитнес-функция: 199.9976200510 | Длина маршрута: 4
Удаление максимального по удалённости: [4690887573, 10133830491, 11992618965, 4888709650]
Удаление максимального по удалённости: [5334043009, 13291158354, 10951537258, 2536037319]
Удаление минимального по интересности: [4690887573, 10133830491, 11992618965, 4888709650]
Удаление максимального по удалённости: [4690887573, 10133830491, 11992618965, 4888709650]
Удаление минимального по интересности: [4690887573, 10133830491, 11992618965, 4888709650]
Поколение 1 | Лучшая фитнес-функция: -0.0079050681 | Длина маршрута: 4
Удаление максимального по удалённости: [4690887573, 10133830491, 11992618965, 4888709650]
Удаление минимального по интересности: [10844030666, 5334043009, 12786128897, 4935300021]
Удаление максимального по удалённости: [10844030666, 5334043009, 12786128897, 4935300021]
Поколение 2 | Лучшая фитнес-функция: -0.0160164672 | Длина маршрута: 3
Удаление минимального по интересности: [10844030666, 5334043009, 1